In [2]:
import torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. 配置 4-bit 量化参数 (QLoRA 的核心)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,         # 开启双重量化，再省一点显存
    bnb_4bit_quant_type="nf4",              # 使用高精度的 NF4 数据类型
    bnb_4bit_compute_dtype=torch.bfloat16,   # 计算时采用 bf16 精度
    # llm_int8_enable_fp32_cpu_offload=True,
)

In [1]:
# 2. 加载基础模型与分词器
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_path = "/root/autodl-tmp/models/qwen3.6-35b-a3b"


# memory_allocation = {
#     0: "40GiB",       # 限制第 0 张显卡最多使用 30GB（留 2GB 给系统和推理缓存）
#     "cpu": "60GiB"    # 限制最多使用 60GB 的物理内存（根据你的电脑实际内存调整）
# }
# processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)
model = AutoModelForMultimodalLM.from_pretrained(
    model_path, 
    # quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    local_files_only=True,
    # max_memory=memory_allocation
)


/root/autodl-tmp/oki-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 1026/1026 [00:12<00:00, 82.58it/s]


In [14]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

KeyboardInterrupt: 

In [3]:
print(model)

Qwen3_5MoeForConditionalGeneration(
  (model): Qwen3_5MoeModel(
    (visual): Qwen3_5MoeVisionModel(
      (patch_embed): Qwen3_5MoeVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3_5MoeVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3_5MoeVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5MoeVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3_5MoeVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
       

In [6]:
# 找出nn.Linear的module name

# import torch.nn as nn

# # 自动解析模型并提取所有 Linear 层的名字
# target_modules = set()
# for name, module in model.named_modules():
#     # 相当于第一步：判断是不是 Linear 层
#     if isinstance(module, nn.Linear): 
#         # 相当于第二步：提取名字 (把诸如 'layers.0.linear_attn.in_proj_qkv' 拆分并取最后一段)
#         layer_name = name.split('.')[-1] 
#         target_modules.add(layer_name)

# print(list(target_modules))


import torch.nn as nn
from collections import defaultdict

VISION_PREFIX = "visual"
LANGUAGE_PREFIX = "language_model"

target_modules = defaultdict(set)

for name, module in model.named_modules():

    if isinstance(module, nn.Linear):

        short_name = name.split(".")[-1]

        if VISION_PREFIX in name.lower():
            target_modules["vision"].add(short_name)

        elif LANGUAGE_PREFIX in name.lower():
            target_modules["language"].add(short_name)


print("Vision nn.Linear modules:")
print(sorted(target_modules["vision"]))

print("\nLanguage nn.Linear modules:")
print(sorted(target_modules["language"]))

Vision nn.Linear modules:
['linear_fc1', 'linear_fc2', 'proj', 'qkv']

Language nn.Linear modules:
['down_proj', 'gate_proj', 'in_proj_a', 'in_proj_b', 'in_proj_qkv', 'in_proj_z', 'k_proj', 'o_proj', 'out_proj', 'q_proj', 'shared_expert_gate', 'up_proj', 'v_proj']


In [4]:
# 打印moe layer class names (for deepspeed config)
from collections import Counter

moe_classes = Counter()

for name, module in model.named_modules():
    cls_name = module.__class__.__name__

    if (
        "moe" in cls_name.lower()
        or "expert" in cls_name.lower()
        or hasattr(module, "experts")
    ):
        moe_classes[cls_name] += 1
        print(f"{name}: {cls_name}")

print("\nMoE-related class counts:")
for cls_name, count in moe_classes.items():
    print(f"{cls_name}: {count}")

: Qwen3_5MoeForConditionalGeneration
model: Qwen3_5MoeModel
model.visual: Qwen3_5MoeVisionModel
model.visual.patch_embed: Qwen3_5MoeVisionPatchEmbed
model.visual.rotary_pos_emb: Qwen3_5MoeVisionRotaryEmbedding
model.visual.blocks.0: Qwen3_5MoeVisionBlock
model.visual.blocks.0.attn: Qwen3_5MoeVisionAttention
model.visual.blocks.0.mlp: Qwen3_5MoeVisionMLP
model.visual.blocks.1: Qwen3_5MoeVisionBlock
model.visual.blocks.1.attn: Qwen3_5MoeVisionAttention
model.visual.blocks.1.mlp: Qwen3_5MoeVisionMLP
model.visual.blocks.2: Qwen3_5MoeVisionBlock
model.visual.blocks.2.attn: Qwen3_5MoeVisionAttention
model.visual.blocks.2.mlp: Qwen3_5MoeVisionMLP
model.visual.blocks.3: Qwen3_5MoeVisionBlock
model.visual.blocks.3.attn: Qwen3_5MoeVisionAttention
model.visual.blocks.3.mlp: Qwen3_5MoeVisionMLP
model.visual.blocks.4: Qwen3_5MoeVisionBlock
model.visual.blocks.4.attn: Qwen3_5MoeVisionAttention
model.visual.blocks.4.mlp: Qwen3_5MoeVisionMLP
model.visual.blocks.5: Qwen3_5MoeVisionBlock
model.visual.bl

In [4]:
for name, module in model.named_modules():
    if any(key in name.lower() for key in ["expert", "moe"]):
        print(name, type(module))

model.language_model.layers.0.mlp.experts <class 'transformers.models.qwen3_5_moe.modeling_qwen3_5_moe.Qwen3_5MoeExperts'>
model.language_model.layers.0.mlp.experts.act_fn <class 'transformers.activations.SiLUActivation'>
model.language_model.layers.0.mlp.shared_expert <class 'transformers.models.qwen3_5_moe.modeling_qwen3_5_moe.Qwen3_5MoeMLP'>
model.language_model.layers.0.mlp.shared_expert.gate_proj <class 'bitsandbytes.nn.modules.Linear4bit'>
model.language_model.layers.0.mlp.shared_expert.up_proj <class 'bitsandbytes.nn.modules.Linear4bit'>
model.language_model.layers.0.mlp.shared_expert.down_proj <class 'bitsandbytes.nn.modules.Linear4bit'>
model.language_model.layers.0.mlp.shared_expert.act_fn <class 'transformers.activations.SiLUActivation'>
model.language_model.layers.0.mlp.shared_expert_gate <class 'bitsandbytes.nn.modules.Linear4bit'>
model.language_model.layers.1.mlp.experts <class 'transformers.models.qwen3_5_moe.modeling_qwen3_5_moe.Qwen3_5MoeExperts'>
model.language_model

In [5]:
for name, param in model.named_parameters():
    if "expert" in name.lower():
        print(name, tuple(param.shape))


model.language_model.layers.0.mlp.experts.gate_up_proj (256, 1024, 2048)
model.language_model.layers.0.mlp.experts.down_proj (256, 2048, 512)
model.language_model.layers.0.mlp.shared_expert.gate_proj.weight (524288, 1)
model.language_model.layers.0.mlp.shared_expert.up_proj.weight (524288, 1)
model.language_model.layers.0.mlp.shared_expert.down_proj.weight (524288, 1)
model.language_model.layers.0.mlp.shared_expert_gate.weight (1024, 1)
model.language_model.layers.1.mlp.experts.gate_up_proj (256, 1024, 2048)
model.language_model.layers.1.mlp.experts.down_proj (256, 2048, 512)
model.language_model.layers.1.mlp.shared_expert.gate_proj.weight (524288, 1)
model.language_model.layers.1.mlp.shared_expert.up_proj.weight (524288, 1)
model.language_model.layers.1.mlp.shared_expert.down_proj.weight (524288, 1)
model.language_model.layers.1.mlp.shared_expert_gate.weight (1024, 1)
model.language_model.layers.2.mlp.experts.gate_up_proj (256, 1024, 2048)
model.language_model.layers.2.mlp.experts.dow

In [9]:
import bitsandbytes as bnb

bnb_linear4_count = 0
non_bnb_linear4_count = 0

for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        bnb_linear4_count += 1
    else:
        non_bnb_linear4_count += 1

print("bnb Linear4bit:", bnb_linear4_count)
print("non_bnb_linear4_count:", non_bnb_linear4_count)

bnb Linear4bit: 460
non_bnb_linear4_count: 689


In [7]:
from collections import defaultdict

stats = defaultdict(int)

for name, param in model.named_parameters():
    key = (str(param.device), str(param.dtype), type(param).__name__)
    stats[key] += param.numel()

for key, numel in sorted(stats.items(), key=lambda x: x[1], reverse=True):
    print(
        key,
        f"{numel / 1e9:.3f} B params",
        f"raw estimate: {numel * param.element_size() / 1024**3:.2f} GiB",
    )

('cuda:0', 'torch.bfloat16', 'Parameter') 19.048 B params raw estimate: 35.48 GiB
('meta', 'torch.bfloat16', 'Parameter') 14.208 B params raw estimate: 26.46 GiB
('cuda:0', 'torch.uint8', 'Params4bit') 0.628 B params raw estimate: 1.17 GiB
('meta', 'torch.uint8', 'Params4bit') 0.297 B params raw estimate: 0.55 GiB


In [12]:
from collections import defaultdict

stats = defaultdict(lambda: {
    "numel": 0,
    "bytes": 0,
})

for name, param in model.named_parameters():
    key = (
        str(param.device),
        str(param.dtype),
        type(param).__name__,
    )

    stats[key]["numel"] += param.numel()

    if param.device.type != "meta":
        stats[key]["bytes"] += (
            param.numel() * param.element_size()
        )

for key, value in sorted(
    stats.items(),
    key=lambda x: x[1]["numel"],
    reverse=True,
):
    print(
        key,
        f"{value['numel'] / 1e9:.3f} B params",
        f"resident tensor estimate: "
        f"{value['bytes'] / 1024**3:.2f} GiB",
    )

('cuda:0', 'torch.bfloat16', 'Parameter') 19.048 B params resident tensor estimate: 35.48 GiB
('meta', 'torch.bfloat16', 'Parameter') 14.208 B params resident tensor estimate: 0.00 GiB
('cuda:0', 'torch.uint8', 'Params4bit') 0.628 B params resident tensor estimate: 0.59 GiB
('meta', 'torch.uint8', 'Params4bit') 0.297 B params resident tensor estimate: 0.00 GiB


In [8]:
for name, param in model.named_parameters():
    if "expert" in name.lower():
        print(
            name,
            tuple(param.shape),
            param.dtype,
            param.device,
            type(param),
        )

model.language_model.layers.0.mlp.experts.gate_up_proj (256, 1024, 2048) torch.bfloat16 cuda:0 <class 'torch.nn.parameter.Parameter'>
model.language_model.layers.0.mlp.experts.down_proj (256, 2048, 512) torch.bfloat16 cuda:0 <class 'torch.nn.parameter.Parameter'>
model.language_model.layers.0.mlp.shared_expert.gate_proj.weight (524288, 1) torch.uint8 cuda:0 <class 'bitsandbytes.nn.modules.Params4bit'>
model.language_model.layers.0.mlp.shared_expert.up_proj.weight (524288, 1) torch.uint8 cuda:0 <class 'bitsandbytes.nn.modules.Params4bit'>
model.language_model.layers.0.mlp.shared_expert.down_proj.weight (524288, 1) torch.uint8 cuda:0 <class 'bitsandbytes.nn.modules.Params4bit'>
model.language_model.layers.0.mlp.shared_expert_gate.weight (1024, 1) torch.uint8 cuda:0 <class 'bitsandbytes.nn.modules.Params4bit'>
model.language_model.layers.1.mlp.experts.gate_up_proj (256, 1024, 2048) torch.bfloat16 cuda:0 <class 'torch.nn.parameter.Parameter'>
model.language_model.layers.1.mlp.experts.down_

In [10]:
from collections import defaultdict
from typing import Any

import torch


def format_num_params(num_params: int) -> str:
    """将参数量格式化为 K/M/B。"""
    if num_params >= 1_000_000_000:
        return f"{num_params / 1_000_000_000:.3f} B"
    if num_params >= 1_000_000:
        return f"{num_params / 1_000_000:.3f} M"
    if num_params >= 1_000:
        return f"{num_params / 1_000:.3f} K"
    return str(num_params)


def format_bytes(num_bytes: int) -> str:
    """将字节数格式化为 KiB/MiB/GiB。"""
    gib = 1024**3
    mib = 1024**2
    kib = 1024

    if num_bytes >= gib:
        return f"{num_bytes / gib:.3f} GiB"
    if num_bytes >= mib:
        return f"{num_bytes / mib:.3f} MiB"
    if num_bytes >= kib:
        return f"{num_bytes / kib:.3f} KiB"
    return f"{num_bytes} B"


def normalize_device(tensor: torch.Tensor) -> str:
    if tensor.device.type == "cuda":
        index = tensor.device.index
        if index is None:
            index = torch.cuda.current_device()
        return f"cuda:{index}"

    return str(tensor.device)


def tensor_storage_info(tensor: torch.Tensor) -> tuple[Any, int]:
    """
    返回：
    - storage 的唯一标识
    - 实际底层 storage 字节数

    用 storage 标识去重 tied weights / shared storage。
    """
    if tensor.device.type == "meta":
        return None, 0

    try:
        storage = tensor.untyped_storage()

        storage_key = (
            tensor.device.type,
            tensor.device.index,
            storage.data_ptr(),
            storage.nbytes(),
        )
        return storage_key, storage.nbytes()

    except (RuntimeError, NotImplementedError):
        # 某些特殊 tensor subclass 可能无法访问 storage。
        # 退化为 numel × element_size。
        storage_key = (
            tensor.device.type,
            tensor.device.index,
            id(tensor),
        )
        return storage_key, tensor.numel() * tensor.element_size()


def get_tensor_class_name(tensor: torch.Tensor) -> str:
    return type(tensor).__name__


def collect_model_memory_stats(model):
    """
    分别统计 parameters 和 buffers。

    返回：
      param_stats:
        key = (device, dtype, tensor_class)
      buffer_stats:
        key = (device, dtype)
    """
    param_stats = defaultdict(
        lambda: {
            "tensor_entries": 0,
            "logical_numel": 0,
            "unique_numel": 0,
            "storage_bytes": 0,
            "trainable_numel": 0,
            "meta_numel": 0,
        }
    )

    buffer_stats = defaultdict(
        lambda: {
            "tensor_entries": 0,
            "logical_numel": 0,
            "storage_bytes": 0,
            "meta_numel": 0,
        }
    )

    # 全局去重，避免 tied weights 被计入多次
    seen_param_objects = set()
    seen_param_storages = set()
    seen_buffer_storages = set()

    # remove_duplicate=False 可以看到所有命名入口，
    # 然后由我们自己按对象/storage 去重。
    try:
        named_parameters = model.named_parameters(remove_duplicate=False)
    except TypeError:
        named_parameters = model.named_parameters()

    for name, param in named_parameters:
        device = normalize_device(param)
        dtype = str(param.dtype).replace("torch.", "")
        tensor_class = get_tensor_class_name(param)

        key = (device, dtype, tensor_class)
        stat = param_stats[key]

        stat["tensor_entries"] += 1
        stat["logical_numel"] += param.numel()

        if param.device.type == "meta":
            stat["meta_numel"] += param.numel()
            continue

        # 同一个 Parameter 对象只计算一次参数量
        param_object_id = id(param)

        if param_object_id not in seen_param_objects:
            seen_param_objects.add(param_object_id)
            stat["unique_numel"] += param.numel()

            if param.requires_grad:
                stat["trainable_numel"] += param.numel()

        # 实际存储按 storage 去重
        storage_key, storage_bytes = tensor_storage_info(param)

        if (
            storage_key is not None
            and storage_key not in seen_param_storages
        ):
            seen_param_storages.add(storage_key)
            stat["storage_bytes"] += storage_bytes

    try:
        named_buffers = model.named_buffers(remove_duplicate=False)
    except TypeError:
        named_buffers = model.named_buffers()

    for name, buffer in named_buffers:
        device = normalize_device(buffer)
        dtype = str(buffer.dtype).replace("torch.", "")
        key = (device, dtype)

        stat = buffer_stats[key]
        stat["tensor_entries"] += 1
        stat["logical_numel"] += buffer.numel()

        if buffer.device.type == "meta":
            stat["meta_numel"] += buffer.numel()
            continue

        storage_key, storage_bytes = tensor_storage_info(buffer)

        if (
            storage_key is not None
            and storage_key not in seen_buffer_storages
        ):
            seen_buffer_storages.add(storage_key)
            stat["storage_bytes"] += storage_bytes

    return param_stats, buffer_stats


def print_model_memory_report(model):
    param_stats, buffer_stats = collect_model_memory_stats(model)

    print("=" * 110)
    print("PARAMETERS")
    print("=" * 110)

    header = (
        f"{'Device':<12}"
        f"{'dtype':<12}"
        f"{'Tensor class':<18}"
        f"{'Unique params':>18}"
        f"{'Trainable':>16}"
        f"{'Storage':>16}"
        f"{'Meta params':>16}"
    )
    print(header)
    print("-" * 110)

    total_unique_params = 0
    total_trainable_params = 0
    total_param_storage = 0
    total_meta_params = 0

    device_param_totals = defaultdict(
        lambda: {
            "numel": 0,
            "trainable": 0,
            "storage_bytes": 0,
            "meta_numel": 0,
        }
    )

    for key in sorted(param_stats):
        device, dtype, tensor_class = key
        stat = param_stats[key]

        print(
            f"{device:<12}"
            f"{dtype:<12}"
            f"{tensor_class:<18}"
            f"{format_num_params(stat['unique_numel']):>18}"
            f"{format_num_params(stat['trainable_numel']):>16}"
            f"{format_bytes(stat['storage_bytes']):>16}"
            f"{format_num_params(stat['meta_numel']):>16}"
        )

        total_unique_params += stat["unique_numel"]
        total_trainable_params += stat["trainable_numel"]
        total_param_storage += stat["storage_bytes"]
        total_meta_params += stat["meta_numel"]

        device_param_totals[device]["numel"] += stat["unique_numel"]
        device_param_totals[device]["trainable"] += stat["trainable_numel"]
        device_param_totals[device]["storage_bytes"] += stat["storage_bytes"]
        device_param_totals[device]["meta_numel"] += stat["meta_numel"]

    print("-" * 110)
    print(
        f"{'TOTAL':<42}"
        f"{format_num_params(total_unique_params):>18}"
        f"{format_num_params(total_trainable_params):>16}"
        f"{format_bytes(total_param_storage):>16}"
        f"{format_num_params(total_meta_params):>16}"
    )

    print()
    print("=" * 82)
    print("PARAMETER TOTALS BY DEVICE")
    print("=" * 82)
    print(
        f"{'Device':<16}"
        f"{'Parameters':>18}"
        f"{'Trainable':>18}"
        f"{'Storage':>16}"
        f"{'Meta params':>16}"
    )
    print("-" * 82)

    for device, stat in sorted(device_param_totals.items()):
        print(
            f"{device:<16}"
            f"{format_num_params(stat['numel']):>18}"
            f"{format_num_params(stat['trainable']):>18}"
            f"{format_bytes(stat['storage_bytes']):>16}"
            f"{format_num_params(stat['meta_numel']):>16}"
        )

    print()
    print("=" * 82)
    print("BUFFERS")
    print("=" * 82)
    print(
        f"{'Device':<16}"
        f"{'dtype':<16}"
        f"{'Elements':>18}"
        f"{'Storage':>16}"
        f"{'Meta elements':>16}"
    )
    print("-" * 82)

    total_buffer_storage = 0
    device_buffer_totals = defaultdict(int)

    for key in sorted(buffer_stats):
        device, dtype = key
        stat = buffer_stats[key]

        print(
            f"{device:<16}"
            f"{dtype:<16}"
            f"{format_num_params(stat['logical_numel']):>18}"
            f"{format_bytes(stat['storage_bytes']):>16}"
            f"{format_num_params(stat['meta_numel']):>16}"
        )

        total_buffer_storage += stat["storage_bytes"]
        device_buffer_totals[device] += stat["storage_bytes"]

    print("-" * 82)
    print(
        f"{'TOTAL BUFFER STORAGE':<50}"
        f"{format_bytes(total_buffer_storage):>16}"
    )

    print()
    print("=" * 82)
    print("MODEL STORAGE BY DEVICE: PARAMETERS + BUFFERS")
    print("=" * 82)

    all_devices = set(device_param_totals) | set(device_buffer_totals)

    for device in sorted(all_devices):
        param_bytes = device_param_totals[device]["storage_bytes"]
        buffer_bytes = device_buffer_totals[device]
        total_bytes = param_bytes + buffer_bytes

        print(
            f"{device:<12} "
            f"parameters={format_bytes(param_bytes):>12}, "
            f"buffers={format_bytes(buffer_bytes):>12}, "
            f"total={format_bytes(total_bytes):>12}"
        )

    print()
    print("=" * 82)
    print("CUDA RUNTIME MEMORY")
    print("=" * 82)

    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return

    for device_index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(device_index)

        allocated = torch.cuda.memory_allocated(device_index)
        reserved = torch.cuda.memory_reserved(device_index)
        peak_allocated = torch.cuda.max_memory_allocated(device_index)
        peak_reserved = torch.cuda.max_memory_reserved(device_index)
        total_memory = properties.total_memory

        print(f"cuda:{device_index} — {properties.name}")
        print(f"  Model tensor storage : "
              f"{format_bytes(device_param_totals[f'cuda:{device_index}']['storage_bytes'] + device_buffer_totals[f'cuda:{device_index}'])}")
        print(f"  CUDA allocated       : {format_bytes(allocated)}")
        print(f"  CUDA reserved        : {format_bytes(reserved)}")
        print(f"  Peak allocated       : {format_bytes(peak_allocated)}")
        print(f"  Peak reserved        : {format_bytes(peak_reserved)}")
        print(f"  Device capacity      : {format_bytes(total_memory)}")
        print(f"  Currently free       : {format_bytes(total_memory - reserved)}")


print_model_memory_report(model)

PARAMETERS
Device      dtype       Tensor class           Unique params       Trainable         Storage     Meta params
--------------------------------------------------------------------------------------------------------------
cuda:0      bfloat16    Parameter                   19.048 B        19.048 B      35.480 GiB               0
cuda:0      uint8       Params4bit                 628.397 M               0     599.286 MiB               0
meta        bfloat16    Parameter                          0               0             0 B        14.208 B
meta        uint8       Params4bit                         0               0             0 B       297.027 M
--------------------------------------------------------------------------------------------------------------
TOTAL                                               19.677 B        19.048 B      36.065 GiB        14.505 B

PARAMETER TOTALS BY DEVICE
Device                  Parameters         Trainable         Storage     Meta params


In [11]:
# 汇总 GPU 上未量化参数所属模块
from collections import defaultdict
import torch


def format_params(n: int) -> str:
    if n >= 1_000_000_000:
        return f"{n / 1e9:.3f} B"
    if n >= 1_000_000:
        return f"{n / 1e6:.3f} M"
    return f"{n:,}"


stats = defaultdict(int)

for name, param in model.named_parameters():
    if param.device.type != "cuda":
        continue

    if param.dtype not in {
        torch.bfloat16,
        torch.float16,
        torch.float32,
    }:
        continue

    # 按模块类型分类
    lower_name = name.lower()

    if ".experts." in lower_name:
        category = "routed_experts"
    elif "shared_expert" in lower_name:
        category = "shared_expert"
    elif "visual" in lower_name or "vision" in lower_name:
        category = "vision"
    elif "embed" in lower_name:
        category = "embedding"
    elif "lm_head" in lower_name:
        category = "lm_head"
    elif "self_attn" in lower_name or "attention" in lower_name:
        category = "attention"
    elif "linear_attn" in lower_name or "deltanet" in lower_name:
        category = "deltanet"
    elif "norm" in lower_name:
        category = "norm"
    else:
        category = "other"

    stats[(category, str(param.dtype))] += param.numel()


print(f"{'Category':<25}{'dtype':<18}{'Parameters':>15}")
print("-" * 58)

for (category, dtype), numel in sorted(
    stats.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(
        f"{category:<25}"
        f"{dtype:<18}"
        f"{format_params(numel):>15}"
    )

Category                 dtype                  Parameters
----------------------------------------------------------
routed_experts           torch.bfloat16           18.522 B
embedding                torch.bfloat16          508.559 M
other                    torch.bfloat16           12.059 M
vision                   torch.bfloat16            4.830 M
deltanet                 torch.bfloat16            593,280
attention                torch.bfloat16             49,664
norm                     torch.bfloat16             47,104


In [7]:
from collections import defaultdict
import torch

try:
    from bitsandbytes.nn import Params4bit
except ImportError:
    Params4bit = ()


def classify_category(name: str) -> str:
    name = name.lower()

    if ".experts." in name:
        return "routed_experts"

    if "shared_expert" in name:
        return "shared_expert"

    if "embed_tokens" in name or "embedding" in name:
        return "embedding"

    if "lm_head" in name:
        return "lm_head"

    if "vision" in name or "visual" in name:
        return "vision"

    if (
        "deltanet" in name
        or "linear_attn" in name
        or "linear_attention" in name
    ):
        return "deltanet"

    if (
        "self_attn" in name
        or ".attention." in name
        or ".attn." in name
    ):
        return "attention"

    if "norm" in name:
        return "norm"

    if "router" in name or "gate.weight" in name:
        return "router_gate"

    return "other"


stats = {
    "quantized": defaultdict(
        lambda: {
            "params": 0,
            "bytes": 0,
        }
    ),
    "unquantized": defaultdict(
        lambda: {
            "params": 0,
            "bytes": 0,
        }
    ),
}


for name, param in model.named_parameters():
    category = classify_category(name)
    dtype = str(param.dtype).replace("torch.", "")

    is_quantized = (
        isinstance(param, Params4bit)
        if Params4bit
        else type(param).__name__ == "Params4bit"
    )

    group = "quantized" if is_quantized else "unquantized"
    key = (category, dtype)

    stats[group][key]["params"] += param.numel()
    stats[group][key]["bytes"] += (
        param.numel() * param.element_size()
    )


def print_stats(title: str, data: dict) -> None:
    print()
    print("=" * 82)
    print(title)
    print("=" * 82)
    print(
        f"{'Category':<22}"
        f"{'Parameters':>18}"
        f"{'dtype':>14}"
        f"{'Space':>18}"
    )
    print("-" * 82)

    total_params = 0
    total_bytes = 0

    sorted_items = sorted(
        data.items(),
        key=lambda item: item[1]["bytes"],
        reverse=True,
    )

    for (category, dtype), item in sorted_items:
        numel = item["params"]
        size_bytes = item["bytes"]

        total_params += numel
        total_bytes += size_bytes

        print(
            f"{category:<22}"
            f"{numel / 1e9:>15.3f} B"
            f"{dtype:>14}"
            f"{size_bytes / 1024**3:>15.3f} GiB"
        )

    print("-" * 82)
    print(
        f"{'TOTAL':<22}"
        f"{total_params / 1e9:>15.3f} B"
        f"{'':>14}"
        f"{total_bytes / 1024**3:>15.3f} GiB"
    )


print_stats("QUANTIZED PARAMETERS", stats["quantized"])
print_stats("UNQUANTIZED PARAMETERS", stats["unquantized"])


QUANTIZED PARAMETERS
Category                      Parameters         dtype             Space
----------------------------------------------------------------------------------
deltanet                        0.505 B         uint8          0.471 GiB
vision                          0.221 B         uint8          0.206 GiB
attention                       0.136 B         uint8          0.127 GiB
shared_expert                   0.063 B         uint8          0.059 GiB
----------------------------------------------------------------------------------
TOTAL                           0.925 B                        0.862 GiB

UNQUANTIZED PARAMETERS
Category                      Parameters         dtype             Space
----------------------------------------------------------------------------------
routed_experts                 32.212 B      bfloat16         60.000 GiB
embedding                       0.509 B      bfloat16          0.947 GiB
lm_head                         0.509 B      bfl